In [ ]:
# Loads every new or changed Excel file in this Lakehouse's Files/input folder into Input_table
# (database gitsit-test). Each file's rows get File_Name = the file's name without ".xlsx", e.g.
# "Jet Lending - 2025.09 - $97mm". A file loaded again replaces that file's earlier rows.
# A loaded file is moved to Files/Archive as "<load time> - <file name>". Run by the pipeline
# "Load input files on upload" as soon as a file lands in Files/input, and hourly as a fallback.
# Generated by packages/frontend/scripts/gen-loader.ts - edit there, not here.

import datetime, json, os, re, struct
import pandas as pd
import pyodbc

SERVER = "wssfdnev5nyelmdvw3xumqun3i-pqhpudri7sjejflsu6qgtrtzvm.database.fabric.microsoft.com"
DATABASE = "gitsit-test-e40f6906-4431-48db-8eca-33c1fd246d16"
FOLDER = "/lakehouse/default/Files/input"
ARCHIVE = "Files/Archive"
TABLE_TYPE = "BPL"
TEXT_MAX = 400
# Input_table columns: [Excel header, column, type]; the first column is always loan_id.
COLUMNS = json.loads(r'''[["Loan ID","loan_id","text"],["Pool","pool","text"],["Arrearage","arrearage","decimal"],["As-Of Date","as_of_date","date"],["Sort","sort","decimal"],["Carve","carve","text"],["Group","group","text"],["Dups","dups","text"],["Lien","lien","decimal"],["Prop Address","prop_address","text"],["Prop City","prop_city","text"],["Prop State","prop_state","text"],["Prop Zip","prop_zip","decimal"],["Prop County","prop_county","text"],["Prop MSA","prop_msa","text"],["Prop Type","prop_type","text"],["# Units","num_units","decimal"],["# Properties","num_properties","text"],["BPO Repaired","bpo_repaired","text"],["BPO As-Is","bpo_as_is","decimal"],["BPO QS","bpo_qs","text"],["Cur List Px","cur_list_px","text"],["Value Type","value_type","text"],["Value Date","value_date","text"],["Occupancy Current","occupancy_current","text"],["Occupancy Original","occupancy_original","text"],["Condition","condition","text"],["Comment / Rating","comment_rating","text"],["Location","location","text"],["Annual Taxes","annual_taxes","text"],["HOA","hoa","text"],["Listed","listed","text"],["Age","age","text"],["Year Built","year_built","text"],["Sqft","sqft","text"],["Beds","beds","text"],["Baths","baths","text"],["Disclosure","disclosure","text"],["Orig AIV Date","orig_aiv_date","text"],["Orig ARV Date","orig_arv_date","text"],["Orig AIV","orig_aiv","text"],["Orig ARV","orig_arv","text"],["Orig Rehab Budget","orig_rehab_budget","text"],["Rehab Reserve/Holdback","rehab_reserve_holdback","text"],["Rehab Amount Drawn","rehab_amount_drawn","text"],["Rehab Draws Remaining","rehab_draws_remaining","text"],["GUC / New Construction Flag","guc_new_construction_flag","text"],["BPO Repairs $","bpo_repairs_usd","text"],["Unnamed: 48","unnamed_48","text"],["Unnamed: 49","unnamed_49","text"],["UPB Accruing","upb_accruing","decimal"],["UPB Deferred","upb_deferred","text"],["Escrow Advances","escrow_advances","text"],["Corp Advances","corp_advances","decimal"],["Late Fees & Other","late_fees_and_other","decimal"],["Junior Lien Bal","junior_lien_bal","text"],["Default Rate","default_rate","text"],["Curr Rate","curr_rate","decimal"],["Curr P&I","curr_p_and_i","decimal"],["Curr T&I","curr_t_and_i","text"],["Curr PITI","curr_piti","text"],["Note Next Due Date","note_next_due_date","date"],["Orig Date","orig_date","date"],["Orig First Pay Date","orig_first_pay_date","date"],["Orig Maturity Date","orig_maturity_date","text"],["Orig Sched Term","orig_sched_term","text"],["Orig Amort Term","orig_amort_term","text"],["Orig UPB","orig_upb","text"],["Orig Rate","orig_rate","decimal"],["Orig P&I","orig_p_and_i","text"],["Orig Appraisal","orig_appraisal","text"],["Orig Sales Px","orig_sales_px","text"],["Orig Appr Date","orig_appr_date","text"],["Orig LTV","orig_ltv","text"],["Orig CLTV","orig_cltv","text"],["Orig FICO","orig_fico","text"],["Orig FICO date","orig_fico_date","text"],["Orig Frontend DTI","orig_frontend_dti","text"],["Orig Backend DTI","orig_backend_dti","text"],["Orig DSCR","orig_dscr","text"],["Loan Type","loan_type","text"],["BPL Bridge Flag","bpl_bridge_flag","text"],["BPL Rehab Flag","bpl_rehab_flag","text"],["BPL DSCR Flag","bpl_dscr_flag","text"],["BPL Extension Flag","bpl_extension_flag","text"],["BPL # Extensions","bpl_num_extensions","text"],["BPL Personal Guaranty","bpl_personal_guaranty","text"],["FHA Flag","fha_flag","text"],["VA Flag","va_flag","text"],["GSE Flag","gse_flag","text"],["HUD PSR Flag","hud_psr_flag","text"],["FNMA PSR Flag","fnma_psr_flag","text"],["FRE PSR Flag","fre_psr_flag","text"],["PSR Expiration Date","psr_expiration_date","text"],["Product Type","product_type","text"],["Doc Type","doc_type","text"],["Loan Purpose","loan_purpose","text"],["Mod Flag","mod_flag","text"],["Mod Type","mod_type","text"],["Mod Date","mod_date","date"],["Mod First Pay Date","mod_first_pay_date","text"],["Mod Maturity Date","mod_maturity_date","text"],["Mod Sched Term","mod_sched_term","text"],["Mod Amort Term","mod_amort_term","text"],["PreMod UPB","premod_upb","text"],["Mod UPB","mod_upb","text"],["PreMod Rate","premod_rate","text"],["Mod Rate","mod_rate","text"],["PreMod P&I","premod_p_and_i","text"],["Mod P&I","mod_p_and_i","text"],["Current Sched Term","current_sched_term","decimal"],["Current Amort Term","current_amort_term","text"],["Current First Pay Date","current_first_pay_date","text"],["Current Maturity Date","current_maturity_date","date"],["Unnamed: 114","unnamed_114","text"],["Remaining Term","remaining_term","decimal"],["Rate Type","rate_type","text"],["Interest Calc Method","interest_calc_method","text"],["IO Flag","io_flag","text"],["IO Term","io_term","text"],["IO End","io_end","text"],["Balloon Flag","balloon_flag","text"],["NegAm Flag","negam_flag","text"],["NegAm Pct","negam_pct","text"],["MI Co","mi_co","text"],["MI Pct","mi_pct","text"],["Curr Frontend DTI","curr_frontend_dti","text"],["Curr Backend DTI","curr_backend_dti","text"],["Curr FICO","curr_fico","text"],["Curr FICO date","curr_fico_date","text"],["Employed","employed","text"],["Employment Date","employment_date","text"],["Self Flag","self_flag","text"],["Income","income","text"],["Income Date","income_date","text"],["Unnamed: 135","unnamed_135","text"],["Seller Status","seller_status","text"],["FCL Flag","fcl_flag","text"],["BK Flag","bk_flag","text"],["REO Flag","reo_flag","text"],["FCL Status","fcl_status","text"],["FCL Hold","fcl_hold","text"],["FCL Hold Date","fcl_hold_date","text"],["FCL Contested","fcl_contested","text"],["FCL Milestone","fcl_milestone","text"],["FCL Milestone Date","fcl_milestone_date","text"],["FCL Referral","fcl_referral","text"],["FCL First Legal","fcl_first_legal","text"],["FCL Service Complete","fcl_service_complete","text"],["FCL Judgment","fcl_judgment","text"],["FCL Sale Sched","fcl_sale_sched","text"],["FCL Sale Held","fcl_sale_held","text"],["BK Status","bk_status","text"],["BK Status Date","bk_status_date","text"],["BK Chapter","bk_chapter","text"],["BK Filing Date","bk_filing_date","text"],["BK Post Petition Due Date","bk_post_petition_due_date","text"],["BK POC Filed","bk_poc_filed","text"],["BK POC Bar Date","bk_poc_bar_date","text"],["BK TOC Filed","bk_toc_filed","text"],["BK Relief Request Date","bk_relief_request_date","text"],["BK Relief Granted Date","bk_relief_granted_date","text"],["BK Dismissal Date","bk_dismissal_date","text"],["BK Discharge Date","bk_discharge_date","text"],["BK End Date","bk_end_date","text"],["BK End Reason","bk_end_reason","text"],["BK Count","bk_count","text"],["BK Notes","bk_notes","text"],["LM COVID Active","lm_covid_active","text"],["LM COVID Prior","lm_covid_prior","text"],["LM COVID Deferral Date","lm_covid_deferral_date","text"],["LM COVID Plan Start Date","lm_covid_plan_start_date","text"],["LM COVID Plan End Date","lm_covid_plan_end_date","text"],["LM Flag","lm_flag","text"],["LM Status","lm_status","text"],["LM Date","lm_date","text"],["BWR Reason for Default","bwr_reason_for_default","text"],["BWR Estate or Deceased","bwr_estate_or_deceased","text"],["BWR Litigation","bwr_litigation","text"],["BWR Cease & Desist","bwr_cease_and_desist","text"],["BWR SCRA","bwr_scra","text"],["Unnamed: 181","unnamed_181","text"],["CF$ t-24","cf_usd_t_24","text"],["CF$ t-23","cf_usd_t_23","text"],["CF$ t-22","cf_usd_t_22","text"],["CF$ t-21","cf_usd_t_21","text"],["CF$ t-20","cf_usd_t_20","text"],["CF$ t-19","cf_usd_t_19","text"],["CF$ t-18","cf_usd_t_18","text"],["CF$ t-17","cf_usd_t_17","text"],["CF$ t-16","cf_usd_t_16","text"],["CF$ t-15","cf_usd_t_15","text"],["CF$ t-14","cf_usd_t_14","text"],["CF$ t-13","cf_usd_t_13","text"],["CF$ t-12","cf_usd_t_12","text"],["CF$ t-11","cf_usd_t_11","text"],["CF$ t-10","cf_usd_t_10","text"],["CF$ t-09","cf_usd_t_09","text"],["CF$ t-08","cf_usd_t_08","text"],["CF$ t-07","cf_usd_t_07","text"],["CF$ t-06","cf_usd_t_06","text"],["CF$ t-05","cf_usd_t_05","text"],["CF$ t-04","cf_usd_t_04","text"],["CF$ t-03","cf_usd_t_03","text"],["CF$ t-02","cf_usd_t_02","text"],["CF$ t-01","cf_usd_t_01","text"],["CF$ cur","cf_usd_cur","text"],["Last Payment Date","last_payment_date","text"],["Paystring","paystring","text"],["x30","x30","text"],["x60","x60","text"],["x90","x90","text"],["Due Date t-25","due_date_t_25","text"],["Due Date t-24","due_date_t_24","text"],["Due Date t-23","due_date_t_23","text"],["Due Date t-22","due_date_t_22","text"],["Due Date t-21","due_date_t_21","text"],["Due Date t-20","due_date_t_20","text"],["Due Date t-19","due_date_t_19","text"],["Due Date t-18","due_date_t_18","text"],["Due Date t-17","due_date_t_17","text"],["Due Date t-16","due_date_t_16","text"],["Due Date t-15","due_date_t_15","text"],["Due Date t-14","due_date_t_14","text"],["Due Date t-13","due_date_t_13","text"],["Due Date t-12","due_date_t_12","text"],["Due Date t-11","due_date_t_11","text"],["Due Date t-10","due_date_t_10","text"],["Due Date t-09","due_date_t_09","text"],["Due Date t-08","due_date_t_08","text"],["Due Date t-07","due_date_t_07","text"],["Due Date t-06","due_date_t_06","text"],["Due Date t-05","due_date_t_05","text"],["Due Date t-04","due_date_t_04","text"],["Due Date t-03","due_date_t_03","text"],["Due Date t-02","due_date_t_02","text"],["Due Date t-01","due_date_t_01","text"],["Due Date cur","due_date_cur","text"],["Unnamed: 238","unnamed_238","text"],["Title","title","text"],["Unnamed: 240","unnamed_240","text"],["TPOL","tpol","text"],["Note","note","text"],["Allonge","allonge","text"],["Endorsement","endorsement","text"],["Mortgage","mortgage","text"],["AOM","aom","text"],["Collateral Exceptions","collateral_exceptions","text"],["Lien Enforcement Flag","lien_enforcement_flag","text"],["High Cost Loan","high_cost_loan","text"],["HUD-1","hud_1","text"],["TIL","til","text"],["Texas Cashout","texas_cashout","text"],["Modification","modification","text"],["ATR","atr","text"],["QM","qm","text"],["TRID","trid","text"],["Compliance Exceptions","compliance_exceptions","text"],["Compliance Grade","compliance_grade","text"],["RPL Resale Flag","rpl_resale_flag","text"],["Disclosure 1","disclosure_1","text"],["Disclosure 2","disclosure_2","text"],["Originator","originator","text"],["Current Servicer","current_servicer","text"],["ST Date","st_date","text"],["Prior Servicer","prior_servicer","text"]]''')

def connect():
    token = notebookutils.credentials.getToken("https://database.windows.net/").encode("UTF-16-LE")
    attrs = {1256: struct.pack(f"<I{len(token)}s", len(token), token)}  # SQL_COPT_SS_ACCESS_TOKEN
    return pyodbc.connect(f"Driver={{ODBC Driver 18 for SQL Server}};Server={SERVER},1433;Database={DATABASE};Encrypt=yes;", attrs_before=attrs)

def empty(v):
    return v is None or (isinstance(v, str) and v == "") or (not isinstance(v, str) and pd.isna(v))

def value(v, name, kind):
    """A cell as Input_table stores it (the same rules as the original load)."""
    if empty(v):
        return None
    if isinstance(v, pd.Timestamp):
        v = v.to_pydatetime()
    if kind == "date":
        return v if isinstance(v, datetime.datetime) else None
    if kind == "decimal":
        try:
            return float(v)
        except (TypeError, ValueError):
            return None
    if isinstance(v, datetime.datetime):
        # Excel reads the delinquency bucket "1-30" as January 1930; keep the text people typed.
        return f"{v.month}-{str(v.year)[2:]}" if name == "seller_status" else v.strftime("%Y-%m-%d")
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return str(v)[:TEXT_MAX]

def read_file(path):
    df = pd.read_excel(path, sheet_name=0, header=0, dtype=object)
    headers = [re.sub(r"\s+", " ", str(h)).strip().lower() for h in df.columns]
    index = {h: i for i, h in enumerate(headers)}
    missing = [h for h, _, _ in COLUMNS[1:] if h.lower() not in index]
    rows = []
    for rec in df.itertuples(index=False):
        if empty(rec[0]):
            continue
        row = [value(rec[0], "loan_id", "text")]
        for header, name, kind in COLUMNS[1:]:
            i = index.get(header.lower())
            row.append(value(rec[i], name, kind) if i is not None else None)
        rows.append(row)
    return rows, missing

conn = connect()
cur = conn.cursor()
cur.execute("""IF OBJECT_ID('dbo.Input_files') IS NULL
CREATE TABLE dbo.Input_files (File_Name NVARCHAR(200) NOT NULL PRIMARY KEY, File_Modified DATETIME2 NOT NULL,
  Loans INT NOT NULL, Missing_Columns NVARCHAR(4000) NULL, Loaded_On DATETIME2 NOT NULL)""")
conn.commit()
loaded = {r[0]: r[1] for r in cur.execute("SELECT File_Name, File_Modified FROM dbo.Input_files").fetchall()}

names = ["Table_Type", "File_Name"] + [name for _, name, _ in COLUMNS]
insert = f"INSERT INTO dbo.Input_table ({', '.join('[' + n + ']' for n in names)}) VALUES ({', '.join('?' for _ in names)})"
cur.fast_executemany = True
results = []
os.makedirs(FOLDER, exist_ok=True)
for f in sorted(os.listdir(FOLDER)):
    path = os.path.join(FOLDER, f)
    if f.startswith("~$") or not f.lower().endswith((".xlsx", ".xlsm")) or not os.path.isfile(path):
        continue
    file_name = os.path.splitext(f)[0].strip()[:200]
    modified = datetime.datetime.utcfromtimestamp(os.path.getmtime(path)).replace(microsecond=0)
    if loaded.get(file_name) == modified:
        note = "already loaded"  # loaded before archiving existed: just archive it
    else:
        rows, missing = read_file(path)
        try:
            cur.execute("DELETE FROM dbo.Input_table WHERE File_Name = ?", file_name)
            if rows:
                cur.executemany(insert, [[TABLE_TYPE, file_name] + r for r in rows])
            cur.execute("DELETE FROM dbo.Input_files WHERE File_Name = ?", file_name)
            cur.execute("INSERT INTO dbo.Input_files VALUES (?, ?, ?, ?, SYSUTCDATETIME())",
                        file_name, modified, len(rows), ", ".join(missing) or None)
            conn.commit()
        except Exception as e:
            conn.rollback()
            results.append(f"{file_name}: FAILED - {e}")
            continue
        note = f"{len(rows)} loans" + (f" (missing columns: {', '.join(missing)})" if missing else "")
    # Loaded: move the file to Files/Archive, its name prefixed with the load time so earlier
    # versions of the same file are kept. A file that failed stays in Files/input.
    stamp = datetime.datetime.utcnow().strftime("%Y-%m-%d %H%M%S")
    archived = f"{ARCHIVE}/{stamp} - {f}"
    try:
        notebookutils.fs.mv(f"Files/input/{f}", archived, True)
    except Exception as e:
        archived = f"not archived ({e})"
    results.append(f"{file_name}: {note}; file moved to {archived}")
conn.close()
print("\n".join(results) or "No new or changed files.")
if any("FAILED" in r for r in results):
    raise Exception("\n".join(r for r in results if "FAILED" in r))
